# ML POWERED UTS Prediction of steel

Machine learning workflow for predicting the ultimate tensile strength of steel from metallurgy-inspired synthetic data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import joblib
import math
import random

random.seed(42)
np.random.seed(42)

sns.set(style='whitegrid')
plt.rcParams['figure.dpi'] = 120


In [ ]:
def generate_steel_dataset(n_samples=800, random_state=42):
    rng = np.random.RandomState(random_state)
    C = rng.uniform(0.05, 0.9, n_samples)
    Si = rng.uniform(0.1, 1.5, n_samples)
    Mn = rng.uniform(0.2, 2.0, n_samples)
    Cr = rng.uniform(0.0, 2.5, n_samples)
    Ni = rng.uniform(0.0, 2.0, n_samples)
    Mo = rng.uniform(0.0, 0.6, n_samples)
    austenitizing_temp = rng.uniform(820, 980, n_samples)
    austenitizing_time = rng.uniform(0.5, 2.5, n_samples)
    cooling_rate = rng.uniform(0.5, 200, n_samples)
    temper_temp = rng.choice([200, 350, 450, 600], size=n_samples, p=[0.25, 0.3, 0.3, 0.15])
    grain_size = rng.uniform(2, 60, n_samples)

    sigma0 = 200.0
    comp_strength = 250 * C + 20 * Si + 15 * Mn + 18 * Cr + 12 * Ni + 40 * Mo
    hall_petch = 400.0 / np.sqrt(np.maximum(grain_size, 0.1))
    quench_effect = 120 * (1 / (1 + np.exp(-0.02 * (cooling_rate - 20))))
    temper_loss = np.where(temper_temp < 300, 0, np.where(temper_temp < 400, 40, np.where(temper_temp < 500, 80, 140)))
    heat_effect = -0.08 * (austenitizing_temp - 900) + 5 * np.log1p(austenitizing_time)
    noise = rng.normal(0, 25, n_samples)
    uts = sigma0 + comp_strength + hall_petch + quench_effect + heat_effect - temper_loss + noise
    uts = np.clip(uts, 100, 2000)

    return pd.DataFrame({
        'C_wt%': np.round(C, 3),
        'Si_wt%': np.round(Si, 3),
        'Mn_wt%': np.round(Mn, 3),
        'Cr_wt%': np.round(Cr, 3),
        'Ni_wt%': np.round(Ni, 3),
        'Mo_wt%': np.round(Mo, 3),
        'AustenTemp_C': np.round(austenitizing_temp, 1),
        'AustenTime_hr': np.round(austenitizing_time, 3),
        'CoolingRate_Cps': np.round(cooling_rate, 3),
        'TemperTemp_C': temper_temp,
        'GrainSize_um': np.round(grain_size, 3),
        'UTS_MPa': np.round(uts, 3)
    })

df = generate_steel_dataset()
df.head()


In [ ]:
display(df.describe().T)

plt.figure(figsize=(8, 4))
sns.histplot(df['UTS_MPa'], bins=30, kde=True)
plt.axvline(df['UTS_MPa'].median(), color='k', linestyle='--', label=f"median {df['UTS_MPa'].median():.1f} MPa")
plt.title('UTS distribution (synthetic)')
plt.xlabel('UTS (MPa)')
plt.legend()
plt.show()

corr_with_target = df.corr(numeric_only=True)['UTS_MPa'].drop('UTS_MPa').sort_values(ascending=False)
plt.figure(figsize=(8, 4))
sns.barplot(x=corr_with_target.values, y=corr_with_target.index)
plt.title('Correlation with UTS')
plt.xlabel('Pearson r')
plt.show()


In [ ]:
features = ['C_wt%', 'Si_wt%', 'Mn_wt%', 'Cr_wt%', 'Ni_wt%', 'Mo_wt%', 'AustenTemp_C', 'AustenTime_hr', 'CoolingRate_Cps', 'TemperTemp_C', 'GrainSize_um']
target = 'UTS_MPa'

X = pd.get_dummies(df[features].copy(), columns=['TemperTemp_C'], prefix='Temp')
y = df[target].copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
numeric_cols = ['C_wt%', 'Si_wt%', 'Mn_wt%', 'Cr_wt%', 'Ni_wt%', 'Mo_wt%', 'AustenTemp_C', 'AustenTime_hr', 'CoolingRate_Cps', 'GrainSize_um']
X_train_scaled = X_train.copy()
X_train_scaled.loc[:, numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled = X_test.copy()
X_test_scaled.loc[:, numeric_cols] = scaler.transform(X_test[numeric_cols])

def print_regression_metrics(y_true, y_pred, name='Model'):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f'{name} -> MAE: {mae:.2f} MPa, RMSE: {rmse:.2f} MPa, R2: {r2:.3f}')

lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)
print_regression_metrics(y_test, y_pred_lr, 'LinearRegression')

rf = RandomForestRegressor(n_estimators=250, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print_regression_metrics(y_test, y_pred_rf, 'RandomForestRegressor')

results_df = pd.DataFrame([
    {'model': 'LinearRegression', 'mae': mean_absolute_error(y_test, y_pred_lr), 'rmse': math.sqrt(mean_squared_error(y_test, y_pred_lr)), 'r2': r2_score(y_test, y_pred_lr)},
    {'model': 'RandomForestRegressor', 'mae': mean_absolute_error(y_test, y_pred_rf), 'rmse': math.sqrt(mean_squared_error(y_test, y_pred_rf)), 'r2': r2_score(y_test, y_pred_rf)},
]).sort_values('r2', ascending=False)
results_df


In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_rf, color='hotpink', s=18)
axis_min = min(y_test.min(), y_pred_rf.min())
axis_max = max(y_test.max(), y_pred_rf.max())
axis_line = np.linspace(axis_min, axis_max, 1000)
plt.plot(axis_line, axis_line, color='black', linewidth=1)
plt.title('UTS Prediction - RandomForestRegressor')
plt.xlabel('Actual UTS (MPa)')
plt.ylabel('Predicted UTS (MPa)')
plt.tight_layout()
plt.show()


@Veena Sahu